# Results from the saved reports

This notebook rebuilds the tables in the write-up from `reports/`. It needs no GPU and no model weights.

In [1]:
import json
from pathlib import Path

import pandas as pd

REPORTS = Path("../reports")
LANGUAGES = ["en", "te", "bn", "sw"]


def load(name):
    return json.loads((REPORTS / name).read_text(encoding="utf-8"))


def effect(report, rule="joint"):
    rows = {}
    for metric, value in report["audit"][rule].items():
        rows[metric] = {"change, pts": 100 * value["delta"], "low": 100 * value["ci"][0], "high": 100 * value["ci"][1], "p": value["p_value"]}
    return pd.DataFrame(rows).T.round(4)


def per_language(report, rule="joint"):
    rows = {}
    for code in LANGUAGES:
        row = report["languages"][code]
        guard = row["guard"][rule]
        rows[code] = {
            "n": row["n"],
            "wrong, always answer": row["wrong_if_always_answering"],
            "wrong, guard": guard["joint_risk"],
            "answered": guard["coverage"],
            "wrong among answered": guard["selective_risk"],
            "guard AUROC": row["scores"]["guard"]["auroc"],
        }
    return pd.DataFrame(rows).T.round(3)


test = load("final/rag_test.json")
global_test = load("final/global_test.json")
health = load("final/health_big.json")
strict = load("final/health_strict.json")


## Locked test set, Tiny Aya Fire

The joint rule targets P(answered and wrong) at most 10%.

In [2]:
print("thresholds", test["thresholds"], "n", test["n"])
effect(test)

thresholds {'joint': 0.4473159266863557, 'selective': 0.363219063540588} n 1721


,"change, pts",low,high,p
wrong_delivered,-42.0686,-44.3928,-39.8024,0.0001
right_delivered,-11.6793,-13.1900,-10.1685,0.0001
answered,-53.7478,-56.0721,-51.4236,0.0001


In [3]:
per_language(test)

,n,"wrong, always answer","wrong, guard",answered,wrong among answered,guard AUROC
en,440.0,0.457,0.059,0.445,0.133,0.877
te,669.0,0.489,0.063,0.490,0.128,0.910
bn,113.0,0.566,0.071,0.398,0.178,0.899
sw,499.0,0.491,0.074,0.455,0.163,0.874


The selective rule targets wrong among answered at most 20%.

In [4]:
effect(test, "selective")

,"change, pts",low,high,p
wrong_delivered,-44.2766,-46.6008,-41.9524,0.0001
right_delivered,-15.8048,-17.5479,-14.0616,0.0001
answered,-60.0813,-62.4056,-57.7571,0.0001


In [5]:
per_language(test, "selective")

,n,"wrong, always answer","wrong, guard",answered,wrong among answered,guard AUROC
en,440.0,0.457,0.043,0.364,0.119,0.877
te,669.0,0.489,0.036,0.439,0.082,0.910
bn,113.0,0.566,0.053,0.345,0.154,0.899
sw,499.0,0.491,0.052,0.389,0.134,0.874


## Closed book against retrieval

In [6]:
pd.DataFrame(test["audit"]["closed_to_rag"]).T[["delta", "ci"]]

,delta,ci
wrong_delivered,-0.348635,"[-0.375363, -0.323649]"
right_delivered,0.348635,"[0.323649, 0.375363]"
answered,0.0,"[0.0, 0.0]"


In [7]:
pd.Series(test["closed_book_wrong"], name="closed-book wrong").round(3)

en    0.727
te    0.948
bn    0.876
sw    0.770
Name: closed-book wrong, dtype: float64

## Separation by method

AUROC for detecting a wrong answer on the locked test set.

In [8]:
pd.DataFrame({c: {m: s["auroc"] for m, s in test["languages"][c]["scores"].items()} for c in LANGUAGES}).round(3)

,en,te,bn,sw
guard,0.877,0.910,0.899,0.874
probe,0.871,0.890,0.902,0.866
similarity,0.723,0.877,0.774,0.742
logprob,0.736,0.582,0.755,0.647
p_true,0.773,0.778,0.808,0.846
agreement,0.740,0.604,0.711,0.691
consensus,0.719,0.558,0.647,0.677


## One global threshold for every method

Wrong and delivered under the joint rule, then coverage.

In [9]:
risk = {c: {m: v["joint"]["joint_risk"] for m, v in test["languages"][c]["global_threshold"].items()} for c in LANGUAGES}
coverage = {c: {m: v["joint"]["coverage"] for m, v in test["languages"][c]["global_threshold"].items()} for c in LANGUAGES}
display(pd.DataFrame(risk).round(3))
pd.DataFrame(coverage).round(3)

,en,te,bn,sw
guard,0.059,0.063,0.071,0.074
probe,0.077,0.057,0.062,0.080
similarity,0.093,0.055,0.097,0.088
logprob,0.041,0.269,0.088,0.056
p_true,0.105,0.118,0.062,0.034
agreement,0.000,0.000,0.000,0.000
consensus,0.002,0.254,0.053,0.134


,en,te,bn,sw
guard,0.445,0.490,0.398,0.455
probe,0.466,0.451,0.372,0.461
similarity,0.364,0.441,0.274,0.329
logprob,0.159,0.599,0.274,0.148
p_true,0.452,0.426,0.204,0.279
agreement,0.000,0.000,0.000,0.000
consensus,0.023,0.546,0.124,0.375


## Second model, Tiny Aya Global

In [10]:
effect(global_test)

,"change, pts",low,high,p
wrong_delivered,-42.3010,-44.6833,-39.9187,0.0001
right_delivered,-10.8658,-12.3765,-9.4131,0.0001
answered,-53.1668,-55.4910,-50.7844,0.0001


In [11]:
per_language(global_test)

,n,"wrong, always answer","wrong, guard",answered,wrong among answered,guard AUROC
en,440.0,0.457,0.082,0.498,0.164,0.861
te,669.0,0.489,0.055,0.493,0.112,0.919
bn,113.0,0.575,0.062,0.372,0.167,0.906
sw,499.0,0.503,0.072,0.431,0.167,0.878


## Domain shift to health, thresholds unchanged

In [12]:
print("all generated questions", health["n"])
display(effect(health))
per_language(health)

all generated questions 1173


,"change, pts",low,high,p
wrong_delivered,-47.1441,-49.9574,-44.2455,0.0001
right_delivered,-10.2302,-12.0205,-8.5251,0.0001
answered,-57.3743,-60.1876,-54.6462,0.0001


,n,"wrong, always answer","wrong, guard",answered,wrong among answered,guard AUROC
en,470.0,0.496,0.066,0.466,0.142,0.902
te,145.0,0.641,0.124,0.386,0.321,0.828
bn,369.0,0.585,0.122,0.415,0.294,0.822
sw,189.0,0.672,0.116,0.381,0.306,0.877


In [13]:
print("strict subset", strict["n"])
display(effect(strict))
per_language(strict)

strict subset 748


,"change, pts",low,high,p
wrong_delivered,-44.3850,-47.9947,-40.7754,0.0001
right_delivered,-8.5561,-10.6952,-6.6845,0.0001
answered,-52.9412,-56.5508,-49.3316,0.0001


,n,"wrong, always answer","wrong, guard",answered,wrong among answered,guard AUROC
en,322.0,0.478,0.075,0.516,0.145,0.910
te,87.0,0.644,0.138,0.448,0.308,0.870
bn,223.0,0.543,0.121,0.448,0.270,0.825
sw,116.0,0.664,0.112,0.405,0.277,0.903


## Judge check with planted wrong answers

In [14]:
print(json.dumps(load("gate_check.json"), indent=1)[:1500])

{
 "en": {
  "n": 60,
  "true_scores": [
   0.85,
   0.8,
   0.95,
   0.8,
   0.95,
   0.8,
   0.8,
   0.95,
   0.8,
   0.8,
   0.95,
   0.8,
   0.95,
   0.95,
   0.85,
   0.85,
   0.8,
   0.85,
   0.95,
   0.85,
   0.95,
   0.85,
   0.95,
   0.8,
   0.8,
   0.8,
   0.8,
   0.95,
   0.95,
   0.8,
   0.95,
   0.95,
   0.8,
   0.85,
   0.8,
   0.95,
   0.95,
   0.8,
   0.85,
   0.8,
   0.85,
   0.8,
   0.9,
   0.85,
   0.9,
   0.9,
   0.85,
   0.8,
   0.85,
   0.95,
   0.95,
   0.85,
   0.8,
   0.8,
   0.8,
   0.95,
   0.8,
   0.8,
   0.8,
   0.9
  ],
  "planted_scores": [
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.8,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.8,
   0.0,
   0.0,
   0.0,
   0.0,
   0.8,
   0.8,
   0.0,
   0.0,
   0.0,
   0.0,
   0.8,
   0.0,
   0.0,
   0.8,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.8,
   0.0,
   0.0,
   0.0

## Latency and cost

In [15]:
pd.DataFrame({k: v for k, v in load("latency.json").items() if isinstance(v, dict)}).T.round(3)

,mean,median,p95
answer_only,0.248,0.223,0.454
guard,0.360,0.331,0.579
p_true,0.347,0.321,0.553
self_consistency,0.935,0.856,1.443
consensus,0.761,0.706,1.185


In [16]:
roi = load("roi.json")
print("input tokens", round(roi["prompt_tokens"]), "output tokens", round(roi["answer_tokens"]), "capacity per month", round(roi["capacity_per_month"]))
display(pd.Series(roi["api_per_1k"], name="USD per 1,000 questions").round(3))
display(pd.Series(roi["break_even_per_month"], name="break-even questions per month").round(0))
pd.DataFrame(roi["month_cost"]).round(0)

input tokens 568 output tokens 6 capacity per month 7298514


gpt-6-luna               0.060
gemini-3.5-flash-lite    0.185
gemini-3.8-flash         0.448
gpt-6.1-sol              1.196
gpt-6-astra              5.979
Name: USD per 1,000 questions, dtype: float64

gpt-6-luna               9768249.0
gemini-3.5-flash-lite    3149998.0
gemini-3.8-flash         1302433.0
gpt-6.1-sol               488412.0
gpt-6-astra                97682.0
Name: break-even questions per month, dtype: float64

,100000,1000000,5000000
self_host,584.0,584.0,584.0
gpt-6-luna,6.0,60.0,299.0
gemini-3.5-flash-lite,19.0,185.0,927.0
gemini-3.8-flash,45.0,448.0,2242.0
gpt-6.1-sol,120.0,1196.0,5979.0
gpt-6-astra,598.0,5979.0,29893.0


## Circuits

In [17]:
overlap = load("overlap.json")
print("heads", overlap["heads"], "top", overlap["top"], "chance", overlap["chance_shared"], "shared by all", overlap["shared_by_all"])
display(pd.DataFrame(overlap["pairs"]).T.round(3))
pd.DataFrame(overlap["peak_block"], index=["first", "second", "third"])

heads 576 top 50 chance 4.3255 shared by all 32


,spearman,shared_top
en-te,0.703,39.0
en-bn,0.746,40.0
en-sw,0.746,43.0
te-bn,0.747,38.0
te-sw,0.741,37.0
bn-sw,0.753,37.0


,en,te,bn,sw
first,23,23,23,23
second,31,31,24,31
third,27,27,31,27
